In [1]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.embeddings import FastEmbedEmbeddings
from rich import print
from IPython.display import Markdown, display
import os 
from dotenv import load_dotenv

c:\Users\richm\OneDrive\Desktop\AI Engineering\LangChain-and-LangGraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
load_dotenv(override=True)
openai_key = os.getenv("OPENAI_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")

In [3]:
# url = "https://365datascience.com/courses/"
url = "https://www.imdb.com/title/tt1190634/?ref_=hm_tenup_t_1"
# file_path = "Business_Analytics.pdf"
file_path = "Formal Paper Assignment Sheet.pdf"

In [4]:
# loader = WebBaseLoader(url)
loader = PyPDFLoader(file_path)

In [5]:
raw_docs = loader.load()

In [6]:
print(raw_docs)

[
    Document(
        metadata={
            'producer': 'PyPDF',
            'creator': 'Microsoft Word',
            'creationdate': '2026-04-12T16:56:17+00:00',
            'author': 'Katie Kornacki',
            'moddate': '2026-04-12T16:56:17+00:00',
            'source': 'Formal Paper Assignment Sheet.pdf',
            'total_pages': 2,
            'page': 0,
            'page_label': '1'
        },
        page_content='EN 345 African American Literature \n \nFormal Paper: \nMemory & the Neo-Slave Narrative \n 
\n \nWe began the semester by reading several examples of slave narratives, the foundational genre for \nAfrican 
American literature. How have 20th- and 21st-century African American writers made use of this \nform, “talking 
back” to these earlier texts in the form of the neo-slave narrative? \n \nFor this paper, you will have the 
opportunity to examine a neo-slave narrative: Octavia Butler’s \nKindred. We will spend some time in class 
(PowerPoint lectures, Discussion Forums, class meetings) \nexploring and discussing the features of neo-slave 
narratives.  \n \n \nThe Prompt:  \nWrite a paper in which you explore the ways that Butler makes use of the 
neo-slave narrative form in \nKindred to re-write and rediscover a significant part (or parts) of the history of 
slavery that was \ndeliberately forgotten and denied.  \n \n \n \nRequirements: \n \nLength: 5-7 pages \nFormat: 
MLA (including formatting and citations) \nTexts: Your paper must discuss Kindred; you are also encouraged to make 
connections between this \nnovel and one or more of the slave narratives we read earlier this semester, either as a
class or for your \nSlave Narrative Presentation. \nCitations: You should follow MLA guidelines and use in-text 
(parenthetical) citations, as well as \ninclude a list of Works Cited. \nDue Dates:  \nPrewriting, Drafting, and 
Revision: 2-page response: Tuesday, 4/21 by 11:59 PM (Week 13) \nRough Draft: Wednesday, 4/29 by 1:00 PM (Week 14) 
\n• Please come to class with a full (4-page or more) rough draft; we will have a \nwriting workshop so come 
prepared \nFinal Draft: Wednesday, 5/6 by 11:59 PM (Final Exam period; in place of a formal exam)'
    ),
    Document(
        metadata={
            'producer': 'PyPDF',
            'creator': 'Microsoft Word',
            'creationdate': '2026-04-12T16:56:17+00:00',
            'author': 'Katie Kornacki',
            'moddate': '2026-04-12T16:56:17+00:00',
            'source': 'Formal Paper Assignment Sheet.pdf',
            'total_pages': 2,
            'page': 1,
            'page_label': '2'
        },
        page_content='Suggestions: \nIn developing your analysis, you may want to keep the following 
features/purposes of the neo-slave \nnarrative in mind: \n  \n• seeks to re-write and rediscover a significant part
of history that was deliberately forgotten and \ndenied \n• seeks reconciliation with a past that still haunts the 
present \n• concerned with the past, but its implications rest in the present as well as the future \n• seeks to 
recover African American culture born and developed in a plantation economy and \nculture \n• this culture kept 
enslaved people from mental slavery, even though they were physically \nenslaved \n• often use vernacular speech, 
songs, and humor \n• sometimes combines traditional forms of storytelling and narrative with elements of black 
folklore \nand magical realism \n• focuses on micro-history \n• presents enslaved persons as cultural/historical 
subjects, not animals/things \n• emphasis on the need to remember \n• focuses on micro-history and history from the
bottom up \n• seeks to “find and expose a truth about the interior life of a people who didn’t write it” and to 
“fill \nthe blanks that the slave narratives left” (Toni Morrison) \n• women’s neo-slave narratives: introduce 
gender issues and questions of sexuality that were \ngenerally absent in the narratives written by men \n \n  
\nFocus Areas:  \n \nA succ

In [7]:
text_splitter = RecursiveCharacterTextSplitter()
documents = text_splitter.split_documents(raw_docs)

In [8]:
embeddings = FastEmbedEmbeddings(model="BAAI/bge-base-en-v1.5")

Store data in a vectorstor

In [9]:
vectorstore = FAISS.from_documents(documents, embeddings)

In [10]:
retriever = vectorstore.as_retriever()

In [11]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    api_key= groq_key,
    model="openai/gpt-oss-120b",
    temperature=0
)

In [12]:
chat_hsitory = []
# query = "Which course on 365DataScience can help me learn AI?"
# query = "Explain what the business analysis group assignment is excepting of the students?"
query = "What is this assignment expecting from students and how should they go about it to get it done in precise manner for a good grade"

In [13]:
relevant_docs = retriever.invoke(query)

In [14]:
context = "\n\n".join(doc.page_content for doc in relevant_docs)

In [15]:
display(Markdown(context))

Suggestions: 
In developing your analysis, you may want to keep the following features/purposes of the neo-slave 
narrative in mind: 
  
• seeks to re-write and rediscover a significant part of history that was deliberately forgotten and 
denied 
• seeks reconciliation with a past that still haunts the present 
• concerned with the past, but its implications rest in the present as well as the future 
• seeks to recover African American culture born and developed in a plantation economy and 
culture 
• this culture kept enslaved people from mental slavery, even though they were physically 
enslaved 
• often use vernacular speech, songs, and humor 
• sometimes combines traditional forms of storytelling and narrative with elements of black folklore 
and magical realism 
• focuses on micro-history 
• presents enslaved persons as cultural/historical subjects, not animals/things 
• emphasis on the need to remember 
• focuses on micro-history and history from the bottom up 
• seeks to “find and expose a truth about the interior life of a people who didn’t write it” and to “fill 
the blanks that the slave narratives left” (Toni Morrison) 
• women’s neo-slave narratives: introduce gender issues and questions of sexuality that were 
generally absent in the narratives written by men 
 
  
Focus Areas:  
 
A successful essay will contain/do the following: 
 
1.) Content & Development: presents a focused, well-developed paper with a clear thesis or 
main argument that is explicitly stated and sustained throughout. Ideas are fully explained, 
nuanced, and consistently support the central claim. Arguments go beyond summary or 
restatement, showing independent reasoning and carful reflection. 
2.) Literary Analysis: Integrates textual evidence—quotations, paraphrases, or precise 
references—effectively and consistently. Literary devices, themes, or textual features are 
analyzed and connected directly to the student’s argument, showing careful, detailed 
engagement with the readings. 
3.) Factual Elements: All references to the text(s), context, or relevant literary/historical 
elements are accurate and clearly support the argument. Facts are precise, contextualized, and 
consistently integrated. 
4.)  Voice & Originality: Writing demonstrates a natural, confident, and engaged voice. Ideas 
are original, thoughtfully expressed, and show independent reasoning, interpretive risk, and 
creativity rather than relying on prepackaged or formulaic phrasing. 
5.) Meet Assignment Requirements: correct page layout and MLA format; in-text citations; 
Works Cited page; at least 5-7 pages typed and double-spaced; meets due-dates

EN 345 African American Literature 
 
Formal Paper: 
Memory & the Neo-Slave Narrative 
 
 
We began the semester by reading several examples of slave narratives, the foundational genre for 
African American literature. How have 20th- and 21st-century African American writers made use of this 
form, “talking back” to these earlier texts in the form of the neo-slave narrative? 
 
For this paper, you will have the opportunity to examine a neo-slave narrative: Octavia Butler’s 
Kindred. We will spend some time in class (PowerPoint lectures, Discussion Forums, class meetings) 
exploring and discussing the features of neo-slave narratives.  
 
 
The Prompt:  
Write a paper in which you explore the ways that Butler makes use of the neo-slave narrative form in 
Kindred to re-write and rediscover a significant part (or parts) of the history of slavery that was 
deliberately forgotten and denied.  
 
 
 
Requirements: 
 
Length: 5-7 pages 
Format: MLA (including formatting and citations) 
Texts: Your paper must discuss Kindred; you are also encouraged to make connections between this 
novel and one or more of the slave narratives we read earlier this semester, either as a class or for your 
Slave Narrative Presentation. 
Citations: You should follow MLA guidelines and use in-text (parenthetical) citations, as well as 
include a list of Works Cited. 
Due Dates:  
Prewriting, Drafting, and Revision: 2-page response: Tuesday, 4/21 by 11:59 PM (Week 13) 
Rough Draft: Wednesday, 4/29 by 1:00 PM (Week 14) 
• Please come to class with a full (4-page or more) rough draft; we will have a 
writing workshop so come prepared 
Final Draft: Wednesday, 5/6 by 11:59 PM (Final Exam period; in place of a formal exam)

In [16]:
history_text = "\n".join(f"User: {q}\nAssistant: {a}" for a, q in chat_hsitory)

In [17]:
prompt = f""" 
Use the context below to answer the question

Conversation history:
{history_text}

context:
{context}

Question:
{query}
"""

In [18]:
response = llm.invoke(prompt)
chat_hsitory.append((query, response.content))

In [19]:
display(Markdown(chat_hsitory[0][1]))

**How to Meet the Expectations of the “Memory & the Neo‑Slave Narrative” Paper  
(Octavia Butler’s *Kindred*) and Earn a Strong Grade**

Below is a step‑by‑step guide that translates the rubric‑style “Focus Areas” and the assignment prompt into concrete actions. Follow it in order, and you’ll hit every requirement the professor is looking for.

---

## 1. Understand What the Professor Is Asking For  

| Prompt Requirement | What It Means in Practice |
|--------------------|---------------------------|
| **“Explore the ways that Butler makes use of the neo‑slave narrative form in *Kindred* to re‑write and rediscover a significant part (or parts) of the history of slavery that was deliberately forgotten and denied.”** | Identify **specific narrative strategies** (e.g., time‑travel, first‑person voice, folk‑song motifs, micro‑history scenes) that Butler borrows from classic slave narratives and show how those strategies **re‑construct hidden histories** (e.g., the lived experience of a mixed‑race woman, the economics of the plantation, gendered violence). |
| **Connect *Kindred* to at least one earlier slave narrative we read.** | Choose a primary source (e.g., Frederick Douglass’s *Narrative*, Harriet Jacobs’s *Incidents in the Life of a Slave Girl*, or any other text you presented in class). Show **direct dialogue** between the two texts—what Butler “talks back” to, expands, or subverts. |
| **5‑7 double‑spaced pages, MLA format, with Works‑Cited.** | Roughly **1,250‑1,750 words** (≈250‑300 words per page). Use 12‑pt Times New Roman, 1‑inch margins, header with last name + page number, and a Works‑Cited page that includes *Kindred* and the slave narrative(s) you cite. |
| **Demonstrate literary analysis, factual accuracy, original voice, and proper mechanics.** | Use **close reading** (quotations + analysis), embed **historical context** (e.g., 19th‑century plantation economics, the “forgotten” aspects of slave law), and write in a **confident, scholarly tone** that goes beyond summary. |

---

## 2. Create a Working Timeline (10‑day plan)

| Day | Task | Deliverable |
|-----|------|-------------|
| **Day 1** | **Read/skim** *Kindred* (focus on the scenes that involve the plantation, Rufus, Dana’s time‑travel, and the “memory” moments). | Highlighted passages + marginal notes. |
| **Day 2** | **Reread** the chosen slave narrative(s). Mark passages that discuss memory, micro‑history, gender, or oral/folk elements. | Annotated PDF or notebook. |
| **Day 3** | **Brainstorm**: Write a list of Butler’s “neo‑slave” techniques (e.g., time‑travel as a metaphor for historical memory, use of vernacular, intergenerational trauma). | 10‑12 bullet points. |
| **Day 4** | **Draft a thesis statement** (1‑2 sentences) that ties together: (a) Butler’s technique, (b) the “forgotten” history she uncovers, (c) the dialogue with the earlier narrative. | One polished thesis. |
| **Day 5** | **Outline** the essay (intro, 3‑4 body sections, conclusion). Include the specific textual evidence you’ll use in each paragraph. | Detailed outline with sub‑points and citation placeholders. |
| **Day 6‑7** | **Write the first draft** (≈4 pages). Focus on getting ideas down; don’t worry about perfect MLA yet. | Rough draft (Word/Google Doc). |
| **Day 8** | **Peer‑review / workshop** (bring the 4‑page draft to class). Take notes on feedback. | List of revision points. |
| **Day 9** | **Revise**: incorporate feedback, tighten analysis, add missing citations, polish MLA formatting. | Revised draft (≈5‑6 pages). |
| **Day 10** | **Proofread** for grammar, MLA consistency, and final word‑count. Submit by the deadline. | Final PDF/Word file. |

---

## 3. Build a Strong Thesis  

A good thesis does three things:

1. **Names the neo‑slave technique(s)** Butler uses.  
2. **States the “forgotten” historical element** she recovers.  
3. **Shows the conversation** with an earlier slave narrative.

**Examples**

*Example 1*  
> In *Kindred*, Octavia Butler employs the neo‑slave narrative device of **time‑travel‑induced micro‑history** to resurrect the **economic agency of enslaved women**—a facet largely omitted from 19th‑century slave narratives such as Harriet Jacobs’s *Incidents in the Life of a Slave Girl*. By juxtaposing Dana’s modern feminist consciousness with the plantation’s gendered labor, Butler rewrites a silenced chapter of slavery and forces readers to confront its lingering impact on contemporary race‑gender relations.

*Example 2*  
> Butler’s *Kindred* “talks back” to Frederick Douglass’s *Narrative* through its **use of vernacular speech and folk song**, but she expands the form by embedding **magical‑realist time slips** that expose the **intergenerational trauma of mixed‑race lineage**, a history deliberately erased from mainstream accounts of slavery.

Pick one that feels most compelling to you, then **test it** against the evidence you have. If you can’t find enough quotations to support a claim, revise the thesis.

---

## 4. Structure the Essay (Suggested 5‑Section Model)

| Section | Purpose | What to Include |
|---------|---------|-----------------|
| **I. Introduction** | Hook → Context → Thesis | • Briefly define “neo‑slave narrative.” <br>• Mention *Kindred* and the chosen slave narrative. <br>• End with your thesis. |
| **II. The Neo‑Slave Narrative Form** | Show you understand the genre. | • Summarize the key features from the “Suggestions” list (memory, micro‑history, vernacular, etc.). <br>• Cite a scholarly source on neo‑slave narratives (e.g., Toni Morrison’s essay, scholarly article). |
| **III. Butler’s Use of Specific Techniques** | Close reading of *Kindred*. | **Paragraphs** (one per technique): <br>1. **Time‑travel as memory‑machine** – quote Dana’s first return (p. xx). <br>2. **Vernacular & song** – quote Rufus’s “old plantation” speech (p. xx). <br>3. **Gendered micro‑history** – scene with Alice (p. xx). <br>4. **Magical realism** – the “pull” of the past (p. xx). <br>Each paragraph: (a) quotation, (b) analysis linking to thesis, (c) brief historical context. |
| **IV. Dialogue with an Earlier Slave Narrative** | Show “talking back.” | • Identify a parallel passage in the slave narrative (e.g., Jacobs’s description of a “master’s house” vs. Butler’s description of the “big house”). <br>• Explain how Butler **mirrors**, **subverts**, or **extends** the earlier text. <br>• Discuss what history is **re‑written** (e.g., the sexual exploitation of mixed‑race children). |
| **V. Conclusion** | Synthesize & look forward. | • Restate how Butler’s neo‑slave strategies recover forgotten history. <br>• Briefly comment on the relevance for today’s “memory politics.” <br>• End with a thought‑provoking sentence (e.g., “In *Kindred*, the past is not a museum but a living wound that still bleeds into the present.”) |

---

## 5. Integrate Textual Evidence Effectively  

1. **Introduce the quote** (who is speaking, what’s happening).  
2. **Present the quotation** (no more than 40 words; longer passages need block‑quote format).  
3. **Analyze**: Explain *how* the language demonstrates the technique and *why* it matters for the forgotten history.  
4. **Connect** back to the thesis.

**Example**

> When Dana first awakens in the antebellum South, she notes, “I was in a house that was not my house, but it was a house that I could not leave” (Butler 23). The forced immobility of the present‑day narrator mirrors the physical confinement of enslaved bodies, turning time travel into a **metaphor for historical memory** that refuses to be escaped. This moment echoes Douglass’s claim that “the mind of the slave is a mind that has been **shackled**” (Douglass 45), yet Butler adds a **gendered layer**: Dana’s modern feminist consciousness is trapped within a patriarchal plantation, exposing a silenced aspect of slave women’s interior lives.

---

## 6. Use Reliable Secondary Sources  

- **Scholarly articles** on neo‑slave narrative theory (e.g., “The Neo‑Slave Narrative and the Politics of Memory” – *African American Review*).  
- **Books** on *Kindred* (e.g., *Octavia Butler’s Kindred: A Critical Companion*).  
- **Historical works** on plantation economics or gendered slavery (to substantiate the “forgotten” history).  

Cite each source in MLA format (author’s last name + page number). Add them to the Works‑Cited page.

---

## 7. MLA Formatting Checklist  

| Item | Correct? |
|------|----------|
| Header (your last name + page number) | ✔ |
| Title (centered, no italics/quotes) | ✔ |
| Double‑spaced, 12‑pt Times New Roman, 1‑inch margins | ✔ |
| In‑text citations (author page) for every quote/paraphrase | ✔ |
| Works‑Cited page (alphabetical, hanging indent) | ✔ |
| Include: *Kindred* (Butler), the slave narrative(s), any scholarly sources | ✔ |
| No extra spaces before/after paragraphs | ✔ |

---

## 8. Writing Voice & Originality  

- **Avoid plot summary**: every paragraph must answer *why* the quoted passage matters to your argument.  
- **Use active, confident language** (“Butler foregrounds…”, “Jacobs reveals…”, not “It seems that…”).  
- **Take a stance**: if you argue that Butler *fails* to fully recover a certain history, explain why and support it with evidence.  
- **Show risk**: connect the novel to a contemporary issue (e.g., “the resurgence of reparations debates”) to demonstrate relevance.

---

## 9. Final Proofreading Checklist  

- **Word count** (≈1,250‑1,750 words).  
- **All quotations** have page numbers.  
- **No dangling citations** (e.g., “(Butler)” without a page).  
- **Spelling of names** (Octavia Butler, Dana, Rufus, Harriet Jacobs, Frederick Douglass).  
- **Consistent tense** (present‑tense literary analysis).  
- **Grammar & punctuation** (run‑on sentences, comma splices).  
- **Works‑Cited** matches in‑text citations (no missing entries).  

---

## 10. Sample Mini‑Outline (to jump‑start your writing)

```
I. Introduction
   A. Hook: “Time travel is a slave’s most terrifying invention…”
   B. Brief definition of neo‑slave narrative.
   C. Introduce *Kindred* and Harriet Jacobs’s *Incidents*.
   D. Thesis (example 1).

II. Neo‑Slave Narrative Features
   A. Memory & micro‑history.
   B. Vernacular & folk song.
   C. Gendered perspective.
   D. Scholarly support (Morrison, etc.).

III. Butler’s Technique #1 – Time‑Travel as Memory Machine
   A. Quote: Dana’s first return (p.23).
   B. Analysis: metaphor for inescapable past.
   C. Historical context: enslaved people’s lack of agency.

IV. Butler’s Technique #2 – Vernacular & Song
   A. Quote: Rufus’s “old plantation” speech (p.57).
   B. Analysis: oral tradition preserving culture.
   C. Compare to Jacobs’s use of “song” to soothe children.

V. Dialogue with Jacobs
   A. Jacobs’s description of “the master’s house” (p.112).
   B. Butler’s parallel scene (p.84) – expands to mixed‑race identity.
   C. What history is recovered? (mixed‑race children’s legal limbo).

VI. Conclusion
   A. Restate thesis.
   B. Summarize how Butler rewrites forgotten history.
   C. Closing thought on the power of memory in contemporary America.
```

---

## 11. Bottom‑Line Tips for a Good Grade  

| Goal | How to Achieve It |
|------|-------------------|
| **Clear, arguable thesis** | Write it early; test it with your evidence. |
| **Strong textual analysis** | Every quote → immediate, specific analysis → link to thesis. |
| **Accurate historical/factual context** | Use at least one reputable secondary source for each